In [1]:
!pip install ortools torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 25.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1

In [2]:
import os
import re
import time
import copy
import random
import math
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.manifold import MDS
from ortools.linear_solver import pywraplp
from joblib import Parallel, delayed

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv

# ==============================================================================
# PHASE 1: DATASET LOADING & ROBUST SCENARIO GENERATION
# ==============================================================================
def load_all_datasets(folder_path):
    distance_matrices = {}
    demand_matrices = {}
    
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' does not exist.")
        return None, None

    print(f"Scanning directory: {folder_path}...\n")
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if not os.path.isfile(file_path): continue
            
        match = re.search(r'\d+', filename)
        if not match: continue
        nodes = int(match.group())
        
        try:
            if filename.startswith(('C-', 'C_')) and filename.endswith('.csv'):
                df = pd.read_csv(file_path, header=None)
                distance_matrices[nodes] = df.values
                print(f"[✓] Loaded Distance Matrix (C) for {nodes} nodes.")
                
            elif filename.startswith('wij'):
                if filename.endswith('.xlsx'): df = pd.read_excel(file_path)
                elif filename.endswith('.csv'): df = pd.read_csv(file_path)
                else: continue
                
                if df.shape[1] > nodes: df = df.iloc[:, 1:]
                demand_matrices[nodes] = df.values
                print(f"[✓] Loaded Demand Matrix (W) for {nodes} nodes.")
        except Exception as e:
            print(f"[!] Warning: Failed to load {filename} - {str(e)}")
            
    return distance_matrices, demand_matrices

def calculate_node_production_attraction(W_matrix):
    return np.sum(W_matrix, axis=1), np.sum(W_matrix, axis=0)

def generate_demand_scenarios(W_matrix, num_scenarios=100, variance_factor=0.2, seed=42):
    np.random.seed(seed)
    n = W_matrix.shape[0]
    W_scenarios = np.zeros((n, n, num_scenarios))
    
    for s in range(num_scenarios):
        noise = np.random.uniform(1.0 - variance_factor, 1.0 + variance_factor, size=(n, n))
        W_scenarios[:, :, s] = W_matrix * noise
        
    return W_scenarios

def generate_subsampled_network(real_distances, real_demands, target_N):
    N = len(real_distances)
    if target_N >= N: return real_distances, real_demands
    indices = np.random.choice(N, target_N, replace=False)
    return real_distances[indices][:, indices], real_demands[indices][:, indices]

# ==============================================================================
# PHASE 2: EXACT SOLVER (BENDERS DECOMPOSITION) & HEURISTICS
# ==============================================================================
def solve_exact_robust_model(distances, W_scenarios, p_hubs, alpha=0.5, beta=0.9, time_limit_mins=5.0, fixed_hubs=None):
    """
    MEMORY CRASH FIX: Implements 3 Advanced OR Techniques:
    1. Variable Reduction (Method 2): Eliminates O(N^4) X_ijkm variables, keeping only O(N^2) z_ik allocations.
    2. Benders Decomposition (Method 3): Master-Subproblem architecture using Integer L-Shaped Cuts.
    3. Memory Bypass (Method 1): 100 Scenarios are evaluated natively in Numpy, bypassing Python objects.
    """
    N = len(distances)
    num_scenarios = W_scenarios.shape[2]
    
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver: return None, float('inf')
    
    # [MASTER PROBLEM SETUP]
    # Extremely lightweight: Only Hubs (y), Allocations (z), and a Cost Surrogate (theta)
    y = {k: solver.IntVar(0, 1, f'y_{k}') for k in range(N)}
    z = {(i, k): solver.IntVar(0, 1, f'z_{i}_{k}') for i in range(N) for k in range(N)}
    theta = solver.NumVar(0, solver.infinity(), 'theta') 
    
    solver.Add(solver.Sum([y[k] for k in range(N)]) == p_hubs)
    
    for i in range(N):
        solver.Add(solver.Sum([z[i, k] for k in range(N)]) == 1)
        for k in range(N):
            solver.Add(z[i, k] <= y[k])
            
    if fixed_hubs is not None:
        for k in range(N):
            solver.Add(y[k] == (1 if k in fixed_hubs else 0))
            
    solver.Minimize(theta)
    
    # Pre-calculate Big-M for Benders Cuts (Strict Upper Bound on Cost)
    max_dist = np.max(distances) * (2 + alpha)
    max_demand = np.sum(np.max(W_scenarios, axis=2))
    big_M = max_dist * max_demand * 1.5 
    
    best_ub = float('inf')
    best_hubs = None
    
    start_time = time.time()
    max_time = time_limit_mins * 60 if time_limit_mins > 0 else float('inf')
    
    iteration = 0
    # [BENDERS MASTER-SUBPROBLEM LOOP]
    while time.time() - start_time < max_time:
        iteration += 1
        
        # Limit solver time per iteration to allow loop to respect max_time
        time_left = max(1, int((max_time - (time.time() - start_time)) * 1000))
        solver.SetTimeLimit(time_left)
        
        status = solver.Solve()
        if status not in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
            break # Proven optimal or infeasible
            
        current_theta = theta.solution_value()
        current_y = sorted([k for k in range(N) if y[k].solution_value() > 0.5])
        
        active_allocations = []
        alloc_idx = np.zeros(N, dtype=int)
        for i in range(N):
            for k in range(N):
                if z[i, k].solution_value() > 0.5:
                    active_allocations.append((i, k))
                    alloc_idx[i] = k
                    
        # [SUBPROBLEM: Vectorized Memory Bypass]
        # Calculate EXACT robust cost natively without OR-Tools objects
        d_i_h = distances[np.arange(N), alloc_idx]
        d_h_h = distances[np.ix_(alloc_idx, alloc_idx)]
        path_costs = d_i_h[:, None] + alpha * d_h_h + d_i_h[None, :]
        
        scenario_costs = np.tensordot(path_costs, W_scenarios, axes=([0, 1], [0, 1]))
        sorted_costs = np.sort(scenario_costs)[::-1]
        beta_S = max(1, math.floor(beta * num_scenarios))
        true_robust_cost = np.mean(sorted_costs[:beta_S])
        
        # Track Best Found Solution
        if true_robust_cost < best_ub:
            best_ub = true_robust_cost
            best_hubs = current_y
            
        # Convergence Check: If Master guessed the cost correctly, it's optimal!
        if current_theta >= true_robust_cost - 1e-4:
            break
            
        # [INTEGER L-SHAPED CUT]
        # Forces Master to update theta if it ever picks this exact allocation again
        cut_expr = true_robust_cost - big_M * (N - solver.Sum([z[i, k] for i, k in active_allocations]))
        solver.Add(theta >= cut_expr)
        
    return best_hubs, best_ub


def evaluate_heuristic_robust_cost(distances, W_scenarios, hubs, alpha=0.5, beta=0.9):
    if not hubs: return float('inf')
    
    N = len(distances)
    alloc = np.argmin(distances[:, hubs], axis=1)
    hubs_assigned = np.array(hubs)[alloc]
    
    d_i_h = distances[np.arange(N), hubs_assigned]
    d_h_h = distances[np.ix_(hubs_assigned, hubs_assigned)]
    path_costs = d_i_h[:, None] + alpha * d_h_h + d_i_h[None, :]
        
    scenario_costs = np.tensordot(path_costs, W_scenarios, axes=([0, 1], [0, 1]))
    sorted_costs = np.sort(scenario_costs)[::-1] 
    
    beta_S = max(1, math.floor(beta * W_scenarios.shape[2]))
    return np.mean(sorted_costs[:beta_S])


def _eval_combo_worker_deterministic(combo, distances, mean_W, alpha):
    N = len(distances)
    alloc = np.argmin(distances[:, combo], axis=1)
    hubs_assigned = np.array(combo)[alloc]
    
    d_i_h = distances[np.arange(N), hubs_assigned]
    d_h_h = distances[np.ix_(hubs_assigned, hubs_assigned)]
    path_costs = d_i_h[:, None] + alpha * d_h_h + d_i_h[None, :]
    
    return np.sum(path_costs * mean_W), list(combo)


def _eval_combo_worker(combo, distances, W_scenarios, alpha, beta):
    cost = evaluate_heuristic_robust_cost(distances, W_scenarios, list(combo), alpha, beta)
    return cost, list(combo)

# ==============================================================================
# PHASE 3: GRAPH NEURAL NETWORK (GNN + GRU HYBRID)
# ==============================================================================
def create_multi_graph_data(distances, demands, labels=None):
    N = len(distances)
    production, attraction = calculate_node_production_attraction(demands)
    spatial_centrality = 1.0 / (np.sum(distances, axis=1) + 1e-6)
    
    x = torch.tensor(np.column_stack((
        production / (np.max(production) or 1.0), 
        attraction / (np.max(attraction) or 1.0),
        spatial_centrality / (np.max(spatial_centrality) or 1.0)
    )), dtype=torch.float)

    edge_index, edge_attr = [], []
    dist_max, dem_max = (np.max(distances) or 1.0), (np.max(demands) or 1.0)
    flow = demands + demands.T
    flow_max = np.max(flow) or 1.0

    for i in range(N):
        for j in range(N):
            if i != j:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]/dist_max, demands[i, j]/dem_max, flow[i, j]/flow_max])

    data = Data(x=x, 
                edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(), 
                edge_attr=torch.tensor(edge_attr, dtype=torch.float))
    if labels is not None: data.y = torch.tensor(labels, dtype=torch.float)
    return data

class HybridGNN_GRU_Ranker(torch.nn.Module):
    def __init__(self, node_in_dim=3, edge_in_dim=3, hidden_dim=32, num_layers=2):
        super(HybridGNN_GRU_Ranker, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(TransformerConv(node_in_dim, hidden_dim, edge_dim=edge_in_dim))
        for _ in range(num_layers - 1):
            self.convs.append(TransformerConv(hidden_dim, hidden_dim, edge_dim=edge_in_dim))
            
        self.gru = torch.nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.fc1 = torch.nn.Linear(hidden_dim, 16)
        self.dropout = torch.nn.Dropout(p=0.2)
        self.fc2 = torch.nn.Linear(16, 1)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        for conv in self.convs:
            x = F.relu(conv(x, edge_index, edge_attr))
            
        x = x.unsqueeze(1)
        x, _ = self.gru(x)
        x = x.squeeze(1)
        
        x = self.dropout(x)
        return self.fc2(F.relu(self.fc1(x))).squeeze()

def train_dl_model(model, train_data_list, val_data_list=None, epochs=200, lr=0.01, patience=15):
    if not train_data_list: return model, float('inf')
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    all_labels = torch.cat([data.y for data in train_data_list])
    pos_weight = ((len(all_labels) - all_labels.sum()) / (all_labels.sum() + 1e-5)).clone().detach().to(device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    eval_data_list = val_data_list if val_data_list is not None and len(val_data_list) > 0 else train_data_list
    best_loss, epochs_no_improve = float('inf'), 0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        model.train()
        for data in train_data_list:
            data = data.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), data.y)
            loss.backward()
            optimizer.step()
            
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for data in eval_data_list:
                data = data.to(device)
                total_val_loss += criterion(model(data), data.y).item()
                
        avg_val_loss = total_val_loss / len(eval_data_list)
        
        if avg_val_loss < best_loss - 1e-4:
            best_loss, epochs_no_improve, best_model_wts = avg_val_loss, 0, copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print(f"   [Early Stopping] Epoch {epoch+1}. Best Val Loss: {best_loss:.4f}")
            break
            
    model.load_state_dict(best_model_wts)
    return model, best_loss

def generate_dynamic_training_data(distance_matrices, demand_matrices, common_nodes, num_graphs=20):
    print(f"\n[AI Training] Generating {num_graphs} dynamic training graphs...")
    training_graphs = []
    
    base_N = max(common_nodes)
    dist, dem = distance_matrices[base_N], demand_matrices[base_N]
    
    for i in range(num_graphs):
        target_N = random.randint(15, 20)
        p_hubs = random.randint(2, 4)
        
        sub_dist, sub_dem = generate_subsampled_network(dist, dem, target_N)
        W_scen = generate_demand_scenarios(sub_dem, num_scenarios=50) 
        exact_hubs, _ = solve_exact_robust_model(sub_dist, W_scen, p_hubs, time_limit_mins=1.0)
        
        if exact_hubs:
            labels = np.zeros(target_N)
            labels[exact_hubs] = 1.0
            data = create_multi_graph_data(sub_dist, sub_dem, labels)
            training_graphs.append(data)
            print(f"   [✓] Graph {i+1}/{num_graphs}: Generated N={target_N}, p={p_hubs}")
            
    return training_graphs

# ==============================================================================
# PHASE 4: ALGORITHMS & HEURISTICS
# ==============================================================================
def get_clustered_candidates(distances, rankings, num_clusters):
    N = len(distances)
    num_clusters = max(1, min(num_clusters, N // 3))
    
    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42, normalized_stress='auto')
    coords = mds.fit_transform(distances)
    
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    return kmeans.fit_predict(coords)

def run_cbs(distances, W_scenarios, demands, p_hubs, alpha=0.5, beta=0.9, num_clusters=5):
    N = len(distances)
    rankings = np.sum(demands, axis=1) + np.sum(demands, axis=0) 
    cluster_labels = get_clustered_candidates(distances, rankings, num_clusters)
    
    candidate_hubs = set()
    for cluster_idx in range(len(set(cluster_labels))):
        nodes = np.where(cluster_labels == cluster_idx)[0]
        sorted_nodes = sorted(nodes, key=lambda x: rankings[x], reverse=True)
        candidate_hubs.update(sorted_nodes[:3])
        
    candidate_hubs.update(np.argsort(rankings)[-min(N, p_hubs * 2):]) 
    candidate_hubs = list(candidate_hubs)
    if len(candidate_hubs) < p_hubs: candidate_hubs = list(np.argsort(rankings)[-p_hubs:])

    combos = list(itertools.combinations(candidate_hubs, p_hubs))
    mean_W = np.mean(W_scenarios, axis=2)
    
    fast_results = Parallel(n_jobs=-1, backend="threading")(
        delayed(_eval_combo_worker_deterministic)(c, distances, mean_W, alpha) for c in combos
    )
    top_combos = [c for cost, c in sorted(fast_results)[:100]] 
    
    results = Parallel(n_jobs=-1, backend="threading")(
        delayed(_eval_combo_worker)(c, distances, W_scenarios, alpha, beta) for c in top_combos
    )
    if results:
        best_cost, best_hubs = min(results, key=lambda x: x[0])
        return sorted(best_hubs), best_cost
    return None, float('inf')


def run_dl_cbs(distances, W_scenarios, dl_rankings, p_hubs, alpha=0.5, beta=0.9, num_clusters=5):
    N = len(distances)
    cluster_labels = get_clustered_candidates(distances, dl_rankings, num_clusters)
    
    candidate_hubs = set()
    for cluster_idx in range(len(set(cluster_labels))):
        nodes = np.where(cluster_labels == cluster_idx)[0]
        best_in_cluster = max(nodes, key=lambda x: dl_rankings[x])
        candidate_hubs.add(best_in_cluster)
        
    sorted_ai_nodes = np.argsort(dl_rankings)[::-1]
    for node in sorted_ai_nodes:
        if len(candidate_hubs) >= max(12, p_hubs * 3): break
        candidate_hubs.add(node)
        
    candidate_hubs = list(candidate_hubs)
    if len(candidate_hubs) < p_hubs: candidate_hubs = list(sorted_ai_nodes[:p_hubs])

    combos = list(itertools.combinations(candidate_hubs, p_hubs))
    mean_W = np.mean(W_scenarios, axis=2)
    
    fast_results = Parallel(n_jobs=-1, backend="threading")(
        delayed(_eval_combo_worker_deterministic)(c, distances, mean_W, alpha) for c in combos
    )
    top_combos = [c for cost, c in sorted(fast_results)[:100]] 

    results = Parallel(n_jobs=-1, backend="threading")(
        delayed(_eval_combo_worker)(c, distances, W_scenarios, alpha, beta) for c in top_combos
    )
    if results:
        best_cost, best_hubs = min(results, key=lambda x: x[0])
        return sorted(best_hubs), best_cost
    return None, float('inf')


def _vnd_local_search(current_hubs, candidate_space, distances, W_scenarios, alpha, beta):
    current_cost = evaluate_heuristic_robust_cost(distances, W_scenarios, current_hubs, alpha, beta)
    improved = True
    
    while improved:
        improved = False
        best_n1_cost, best_n1_hubs = current_cost, current_hubs.copy()
        
        # Neighborhood 1: 1-Swaps
        for i in range(len(current_hubs)):
            for c in candidate_space:
                if c not in current_hubs:
                    temp = current_hubs.copy()
                    temp[i] = c
                    cost = evaluate_heuristic_robust_cost(distances, W_scenarios, temp, alpha, beta)
                    if cost < best_n1_cost:
                        best_n1_cost, best_n1_hubs = cost, temp
                        
        if best_n1_cost < current_cost:
            current_cost, current_hubs = best_n1_cost, best_n1_hubs
            improved = True
            continue 

        # Neighborhood 2: 2-Swaps
        if len(current_hubs) >= 2:
            short_candidates = candidate_space[:15] 
            avail = [c for c in short_candidates if c not in current_hubs]
            for i, j in itertools.combinations(range(len(current_hubs)), 2):
                for c1, c2 in itertools.combinations(avail, 2):
                    temp = current_hubs.copy()
                    temp[i], temp[j] = c1, c2
                    cost = evaluate_heuristic_robust_cost(distances, W_scenarios, temp, alpha, beta)
                    if cost < current_cost:
                        current_cost, current_hubs = cost, temp
                        improved = True
                        break 
                if improved: break
    return current_hubs, current_cost

def _gvns_core(distances, W_scenarios, initial_hubs, candidate_space, alpha, beta, max_iter, epsilon_greedy=False, all_nodes_space=None):
    best_hubs = initial_hubs.copy()
    best_cost = evaluate_heuristic_robust_cost(distances, W_scenarios, best_hubs, alpha, beta)
    
    for _ in range(max_iter):
        k_shake, k_max = 1, max(1, len(best_hubs) - 1)
        
        while k_shake <= k_max:
            test_hubs = best_hubs.copy()
            shake_indices = np.random.choice(len(test_hubs), k_shake, replace=False)
            
            avail = [c for c in (all_nodes_space if epsilon_greedy and np.random.rand() < 0.20 else candidate_space) if c not in test_hubs]
            if len(avail) >= k_shake:
                new_hubs = np.random.choice(avail, k_shake, replace=False)
                for idx, new_hub in zip(shake_indices, new_hubs): test_hubs[idx] = new_hub
                    
            test_hubs, test_cost = _vnd_local_search(test_hubs, candidate_space, distances, W_scenarios, alpha, beta)
            
            if test_cost < best_cost:
                best_cost, best_hubs, k_shake = test_cost, test_hubs, 1 
            else:
                k_shake += 1 
    return sorted(best_hubs), best_cost

def run_gvns(distances, W_scenarios, demands, p_hubs, alpha=0.5, beta=0.9, max_iter=30):
    N = len(distances)
    naive_rankings = np.sum(demands, axis=1) + np.sum(demands, axis=0)
    candidate_space = list(np.argsort(naive_rankings)[-min(N, 40):])
    current_hubs = list(np.random.choice(candidate_space, p_hubs, replace=False))
    return _gvns_core(distances, W_scenarios, current_hubs, candidate_space, alpha, beta, max_iter)

def run_dl_gvns(distances, W_scenarios, dl_rankings, p_hubs, alpha=0.5, beta=0.9, max_iter=30):
    N = len(distances)
    sorted_ai_nodes = np.argsort(dl_rankings)[::-1]
    
    guaranteed_p = min(p_hubs, max(1, p_hubs // 2))
    initial_hubs = list(sorted_ai_nodes[:guaranteed_p])
    wider_pool = list(sorted_ai_nodes[guaranteed_p:min(N, 40)])
    
    if p_hubs - guaranteed_p > 0:
        initial_hubs.extend(np.random.choice(wider_pool, p_hubs - guaranteed_p, replace=False))
        
    candidate_space = list(sorted_ai_nodes[:min(N, 40)])
    return _gvns_core(distances, W_scenarios, initial_hubs, candidate_space, alpha, beta, max_iter, epsilon_greedy=True, all_nodes_space=list(range(N)))

# ==============================================================================
# PHASE 5: EXPORT & PLOTTING
# ==============================================================================
def plot_academic_results(df):
    try:
        sns.set_theme(style="whitegrid")
        
        g1 = sns.catplot(data=df, x='Hubs (p)', y='Robust Cost', hue='Method', col='Nodes', 
                         kind='bar', sharey=False, col_wrap=2, height=4, aspect=1.2, palette="viridis")
        g1.fig.suptitle("Robust Cost Comparison Across Methods by Dataset Size", y=1.02)
        for ax in g1.axes.flat: ax.set_yscale('log')
        g1.savefig("Fig1_Cost_Comparison.png", bbox_inches='tight', dpi=300)
        plt.close()

        g2 = sns.catplot(data=df, x='Hubs (p)', y='Time (s)', hue='Method', col='Nodes', 
                         kind='bar', sharey=False, col_wrap=2, height=4, aspect=1.2, palette="viridis")
        g2.fig.suptitle("Execution Time by Method and Dataset Size", y=1.02)
        for ax in g2.axes.flat: ax.set_yscale('log')
        g2.savefig("Fig2_Time_Comparison.png", bbox_inches='tight', dpi=300)
        plt.close()

        df_heuristics = df[df['Method'] != 'Exact (SCIP)']
        if not df_heuristics.empty:
            plt.figure(figsize=(10, 6))
            sns.lineplot(data=df_heuristics, x='Nodes', y='Gap (%)', hue='Method', marker='o', linewidth=2.5)
            plt.title("Optimality Gap Scaling with Network Size")
            plt.ylabel("Gap from Exact Solver (%)")
            plt.axhline(0, color='black', linestyle='--')
            plt.tight_layout()
            plt.savefig("Fig3_Gap_Scaling.png", dpi=300)
            plt.close()
        
        print("   [✓] Plots generated successfully.")
    except Exception as e:
        print(f"   [!] Error generating plots: {str(e)}")

# ==============================================================================
# MASTER EXECUTION PIPELINE
# ==============================================================================
if __name__ == "__main__":
    print("==================================================")
    print("🚀 AI-AUGMENTED ROBUST HUB LOCATION PIPELINE")
    print("==================================================")
    
    DATA_FOLDER = "/kaggle/input/datasets/infernalss/node-dataset" 
            
    distance_matrices, demand_matrices = load_all_datasets(DATA_FOLDER)
    
    if distance_matrices and demand_matrices:
        common_nodes = sorted(list(set(distance_matrices.keys()) & set(demand_matrices.keys())))
        print(f"\n[Info] Network sizes available for testing: {common_nodes}")
        
        # --- 1 & 2. PREPARE TRAINING DATA & TRAIN MODEL ---
        dataset_path = "/kaggle/input/datasets/infernalss/gnn-training/offline_training_dataset.pt"
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        dl_model = HybridGNN_GRU_Ranker(node_in_dim=3, edge_in_dim=3, hidden_dim=32, num_layers=2).to(device)
        
        if os.path.exists(dataset_path):
            print(f"\n[AI Training] Loading existing training dataset from {dataset_path}")
            training_graphs = torch.load(dataset_path, weights_only=False)
            
            if len(training_graphs) > 4:
                print(f"\n[AI Training] Training Hybrid GNN-GRU Model on {len(training_graphs)} graphs...")
                random.shuffle(training_graphs)
                split_idx = int(len(training_graphs) * 0.8)
                dl_model, final_loss = train_dl_model(dl_model, training_graphs[:split_idx], training_graphs[split_idx:], epochs=150)
                print(f"   [✓] Model training complete. Validation Loss: {final_loss:.4f}")
            else:
                print("[!] Not enough training data. Skipping training.")
        else:
            print(f"\n[!] Offline training data '{dataset_path}' not found. Generating minimal set dynamically...")
            training_graphs = generate_dynamic_training_data(distance_matrices, demand_matrices, common_nodes, num_graphs=10)
            if training_graphs:
                dl_model, _ = train_dl_model(dl_model, training_graphs, epochs=50)
            
        dl_model.eval()
        
        # --- 3. TEST MODEL ON LOADED DATASETS ---
        print("\n" + "="*50)
        print("PHASE 3 & 4: EVALUATING HEURISTICS AND LOGGING RESULTS")
        print("="*50)
        
        results_list = []
        p_values = [2, 3, 4, 5] 
        
        for N in common_nodes:
            distances, demands = distance_matrices[N], demand_matrices[N]
            W_scenarios = generate_demand_scenarios(demands, num_scenarios=100)
            
            with torch.no_grad():
                graph_data = create_multi_graph_data(distances, demands).to(device)
                ai_probs = torch.sigmoid(dl_model(graph_data)).cpu().numpy()
            
            for p in p_values:
                print(f"\n--- Evaluating Nodes={N} | Hubs(p)={p} ---")
                
                start_t = time.time()
                exact_cost, exact_hubs = float('inf'), None
                
                # Benders Exact Solver can now handle N=25 easily and tries N=100 safely.
                print(f"   [Exact] Building exact Benders model (Limit: 5 mins)...")
                try:
                    exact_hubs, exact_cost = solve_exact_robust_model(distances, W_scenarios, p, time_limit_mins=25.0)
                except Exception as e:
                    print(f"   [Exact] Stopped/Error: {e}")
                    
                t_exact = time.time() - start_t
                
                if exact_cost != float('inf') and exact_hubs is not None:
                    results_list.append({
                        'Nodes': N, 'Hubs (p)': p, 'Method': 'Exact (SCIP)', 
                        'Time (s)': round(t_exact, 2), 'Robust Cost': exact_cost, 
                        'Gap (%)': 0.0, 'Selected Hubs': str(exact_hubs)
                    })
                    print(f"   [Exact] Time: {t_exact:.2f}s | Cost: {exact_cost:.2e} | Hubs: {exact_hubs}")
                else:
                    print(f"   [Exact] Timed out or failed. Cost: inf")
                
                def run_and_log(method_name, func, *args):
                    start_t = time.time()
                    hubs, cost = func(*args)
                    t_val = time.time() - start_t
                    
                    if exact_cost != float('inf') and cost != float('inf'):
                        gap = ((cost - exact_cost) / exact_cost) * 100
                    else:
                        gap = 0.0 
                        
                    results_list.append({
                        'Nodes': N, 
                        'Hubs (p)': p, 
                        'Method': method_name, 
                        'Time (s)': round(t_val, 2), 
                        'Robust Cost': cost, 
                        'Gap (%)': round(gap, 2),
                        'Selected Hubs': str(hubs) if hubs else "None"
                    })
                    print(f"   [{method_name}] Time: {t_val:.2f}s | Gap: {gap:.2f}% | Hubs: {hubs}")

                run_and_log('CBS (Standard)', run_cbs, distances, W_scenarios, demands, p)
                run_and_log('DL-CBS (AI)', run_dl_cbs, distances, W_scenarios, ai_probs, p)
                run_and_log('GVNS (Standard)', run_gvns, distances, W_scenarios, demands, p)
                run_and_log('DL-GVNS (AI)', run_dl_gvns, distances, W_scenarios, ai_probs, p)

        # --- 4. EXPORT & REPORT ---
        if results_list:
            df_results = pd.DataFrame(results_list)
            
            cols = ['Nodes', 'Hubs (p)', 'Method', 'Robust Cost', 'Gap (%)', 'Time (s)', 'Selected Hubs']
            df_results = df_results[cols]
            
            csv_filename = "Hub_Location_Experiment_Results.csv"
            df_results.to_csv(csv_filename, index=False)
            print("\n" + "="*50)
            print(f"✅ BATCH PIPELINE COMPLETE!")
            print(f"💾 Detailed master data saved to: '{csv_filename}'.")
            
            plot_academic_results(df_results)
            print("==================================================")
        else:
            print("\n[!] No successful evaluations to log.")

🚀 AI-AUGMENTED ROBUST HUB LOCATION PIPELINE
Scanning directory: /kaggle/input/datasets/infernalss/node-dataset...

[✓] Loaded Demand Matrix (W) for 100 nodes.
[✓] Loaded Distance Matrix (C) for 200 nodes.
[✓] Loaded Demand Matrix (W) for 200 nodes.
[✓] Loaded Demand Matrix (W) for 150 nodes.
[✓] Loaded Distance Matrix (C) for 150 nodes.
[✓] Loaded Distance Matrix (C) for 100 nodes.
[✓] Loaded Demand Matrix (W) for 25 nodes.
[✓] Loaded Distance Matrix (C) for 25 nodes.

[Info] Network sizes available for testing: [25, 100, 150, 200]

[AI Training] Loading existing training dataset from /kaggle/input/datasets/infernalss/gnn-training/offline_training_dataset.pt

[AI Training] Training Hybrid GNN-GRU Model on 560 graphs...
   [Early Stopping] Epoch 34. Best Val Loss: 0.9789
   [✓] Model training complete. Validation Loss: 0.9789

PHASE 3 & 4: EVALUATING HEURISTICS AND LOGGING RESULTS

--- Evaluating Nodes=25 | Hubs(p)=2 ---
   [Exact] Building exact Benders model (Limit: 5 mins)...
   [Exa